# Meta-learning to plan in a maze — Jensen, Hennequin & Mattar (2024)

>Reference: *Jensen, Hennequin & Mattar (2024), Nature Neuroscience* — "**[A recurrent network model of planning explains hippocampal replay and human behavior](https://doi.org/10.1038/s41593-024-01675-7)**" — using the NeuralRNN framework.

**The question.** When a rat pauses at a junction in a maze, hippocampal place
cells "replay" short trajectories through space, and these replay events
predict what the animal does next. What kind of machine would do this, and
why? Jensen and colleagues trained a recurrent neural network with
reinforcement learning on a foraging task where the maze and the reward
location change every episode. The network cannot solve a new maze through
its (frozen) weights — it has to figure each maze out *within* the episode,
through its recurrent dynamics. This is **meta-learning** (in the RL² sense):
learning slow, through gradient descent across episodes; planning fast,
through recurrent activity within an episode.

**What emerges.** The agent learns (i) a *world model* of the maze — an
auxiliary network head that predicts the next state and the reward location —
and (ii) a *think* action that does not move the body but rolls the world
model forward in imagination. The imagined trajectory is fed back into the
network as an input, shifting the hidden state and thereby the policy. The
trained agent chooses to think exactly when thinking helps: at the start of
each foraging round, more so in harder mazes. Its imagined trajectories avoid
walls it has never hit, over-represent the goal, and push the hidden state in
the direction a policy-gradient update would — the network has discovered
something like hippocampal replay, without being built to replay anything.

**Framework mapping** (vs. supervised notebooks):

| layer | this notebook |
|---|---|
| data | a maze **environment** (`neuralrnn.rl.envs.maze.MazeEnv`) generating episodes online |
| model | **`ActorCriticModel`** = GRU core + policy/value head + **world-model aux head** |
| planning | **`WorldModelPlanner`** (imagined rollouts via the aux head, fed back as input) |
| loss | **`A2CPredLoss`** (REINFORCE + value + entropy + world-model prediction) |
| trainer | **`RLTrainer`** (on-policy, episode-aligned rollouts) |

**What is reproduced here** (model side of Figs 2–5; the paper's human
experiments are discussed qualitatively in the text but not re-analyzed):

1. **Behavior (Fig 2):** performance across foraging trials approaches the
   optimal path length; thinking time grows with maze difficulty; the world
   model's belief about the reward sharpens as the agent explores.
2. **Causal role of rollouts (Fig 3):** preventing planning hurts; forcing
   extra rollouts helps; a goal-finding (vs. goal-missing) rollout shifts the
   policy toward the first imagined action.
3. **Replay content (Fig 4):** imagined transitions respect walls they were
   never trained against; goals are over-represented; goal representations
   build up over consecutive rollouts; the agent follows its first imagined
   action precisely when the rollout found the goal.
4. **Mechanism (Fig 5):** the hidden-state update written by rollout feedback
   aligns with the policy-gradient direction for the imagined action — the
   network uses replay to *do* policy improvement in hidden-state space.

Contents:
1. Setup
2. The task: foraging in a maze that changes every episode
3. The agent: a GRU with a policy, a value, and a world model
4. The planner: thinking as imagined trajectories
5. Training (or loading) the agent
6. A trained episode, step by step
7. Behavior of the trained agent (Fig 2)
8. Rollouts causally improve the policy (Fig 3)
9. What replays contain (Fig 4)
10. Why rollouts work: alignment with policy gradient (Fig 5)
11. The world model improves over training (supplementary)
12. Faithfulness notes and deviations


## 1. Setup

Standard imports plus the RL pieces of NeuralRNN. `MODEL_DIR` holds the
trained agent (`models/s2/seed61_full`, one merged run — see Section 5) and
the cached analysis results; `FIG_DIR` collects the figures.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from neuralrnn import (
    AutoModel, RLTrainer, RLTrainingArguments, A2CPredLoss,
    SyncVectorEnv, make_env, collect_episodes, WorldModelPlanner, bind_planners,
)
from neuralrnn.models.actor_critic import ActorCriticConfig, ActorCriticModel
from neuralrnn.models.gated_rnn import GatedRNNConfig
from neuralrnn.rl.envs.maze import MazeEnv, generate_maze, DIRS

MODEL_DIR = "models/s2"
FIG_DIR = "figs/s2"
EVAL_DIR = os.path.join(MODEL_DIR, "eval")   # cached analysis arrays
for d in (MODEL_DIR, FIG_DIR, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 61          # paper's main seed range is 61..65; we train seed 61
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False})
print("neuralrnn ready; models in", os.path.abspath(MODEL_DIR))

## 2. The task: foraging in a maze that changes every episode

**Arena.** A 4×4 grid with *periodic boundaries* (a torus: walking off the
right edge brings you back on the left), so no corner is special. Internal
walls are generated by a recursive-backtracker maze generator, which produces
a fully connected maze, plus 3 extra random openings ("holes") that create
loops. States are indexed $s = 4x + y \in \{0, \dots, 15\}$.

**Reward.** One hidden goal location $p^*$ per episode, uniform over the 16
cells. Moving onto the goal gives reward $r = 1$. The agent is then
*teleported* to a uniformly random non-goal cell on its next action, and can
collect the same reward again — the episode is a sequence of **trials**: an
exploratory trial 1 (goal unknown) followed by exploitation trials
$2, 3, \dots$ in which the optimal behavior is to walk straight back to the
remembered goal.

**Time.** The episode has a budget of $T = 50$ time units. A physical move
costs 1.0; a *think* action (stay in place, run an imagined rollout) costs
0.3. Thinking therefore buys information at roughly a third of the price of
a step — but an episode of pure thinking still ends. Bumping into a wall
costs 1.0 and leaves the agent where it was.

**Actions.** $a \in \{0,1,2,3\}$ for the four cardinal moves and $a = 4$
for *think*.

**Observation** (88 dimensions, what the network sees each step):

| slice | content |
|---|---|
| `0:5` | one-hot of the previous action |
| `5` | previous reward |
| `6` | elapsed time / 50 |
| `7:23` | one-hot current location |
| `23:39` | walls in the +x direction (16 entries, one per cell) |
| `39:55` | walls in the +y direction |
| `55:88` | **planning feedback**: 4×8 one-hot of the imagined actions (8 rollout steps × 4 moves) + one "goal found" flag — all zeros when no rollout happened |

The wall layout is always visible (the animal can see the maze); only the
reward location is hidden. What makes the task a *meta-learning* problem is
that maze and goal are resampled every episode: generalization across
episodes must live in the weights, while maze-specific knowledge (where is
the goal *this* episode?) must live in the hidden state.

`MazeEnv` implements exactly this MDP (ported line-by-line from the Julia
reference, `walls_build.jl` / `maze.jl`).


In [ ]:
# --- an example maze and a random-policy episode ------------------------
env = MazeEnv(seed=3)
obs, info = env.reset()
print("goal:", info["goal_loc"], " start:", info["start_loc"])

def draw_maze(ax, walls, goal=None, path=None, title="", rollouts=None):
    """Draw the 4x4 toroidal maze: cells, walls, goal, and a trajectory.

    walls: (16, 4) array; walls[s, a]=1 blocks direction DIRS[a] from s.
    path: list of state indices visited (drawn as a fading line).
    rollouts: list of (start_state, imagined_state_sequence) in red.
    """
    L = 4
    ax.set_aspect("equal")
    for s in range(16):
        x, y = divmod(s, L)
        for a, (dx, dy) in enumerate(DIRS):
            if walls[s, a] > 0.5:
                nb = (L * ((x + dx) % L) + (y + dy) % L)
                # draw each wall once (only from the cell with smaller index)
                if not (abs(x - (x + dx) % L) > 1 or abs(y - (y + dy) % L) > 1):
                    if dx == 1:
                        ax.plot([x + 0.5, x + 0.5], [y - 0.5, y + 0.5],
                                "k-", lw=2)
                    elif dy == 1:
                        ax.plot([x - 0.5, x + 0.5], [y + 0.5, y + 0.5],
                                "k-", lw=2)
    if goal is not None:
        gx, gy = divmod(goal, L)
        ax.scatter([gx], [gy], marker="*", s=260, color="goldenrod",
                   zorder=5, label="goal")
    if path:
        xs = [s // L for s in path]
        ys = [s % L for s in path]
        for i in range(len(path) - 1):
            # skip the teleport jump and torus wrap-arounds when drawing
            if abs(xs[i+1]-xs[i]) + abs(ys[i+1]-ys[i]) <= 1:
                ax.plot([xs[i], xs[i+1]], [ys[i], ys[i+1]], "o-",
                        color="steelblue", lw=1.5, ms=3, zorder=3)
        ax.scatter([xs[0]], [ys[0]], marker="o", s=90, facecolor="none",
                   edgecolor="steelblue", lw=2, zorder=4, label="start")
    if rollouts:
        for s0, seq in rollouts:
            xs = [s0 // L] + [s // L for s in seq]
            ys = [s0 % L] + [s % L for s in seq]
            ax.plot(xs, ys, "s--", color="firebrick", lw=1.2, ms=3,
                    zorder=4)
    ax.set_xlim(-0.6, L - 0.4)
    ax.set_ylim(-0.6, L - 0.4)
    ax.set_xticks(range(L)); ax.set_yticks(range(L))
    ax.set_title(title)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
        
    
# random-policy episode, recording the path
obs, info = env.reset(seed=0)
path, rews = [env.agent_state], []
done, rng = False, np.random.default_rng(0)
while not done:
    a = int(rng.integers(0, 4))          # random physical moves only
    obs, r, done, _, si = env.step(a)
    path.append(si["agent_loc"]); rews.append(r)

fig, ax = plt.subplots(figsize=(3.6, 3.6))
draw_maze(ax, env.walls, goal=env.goal, path=path,
          title=f"random policy: {sum(r > 0.5 for r in rews)} rewards")
ax.legend(loc="upper right", fontsize=8, frameon=False, bbox_to_anchor=(1.3, 1.0))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_task_example.png"))
plt.show()

A random walker on this maze occasionally stumbles onto the
goal, but the periodic boundaries mean it keeps wandering in circles. The
optimal exploitation policy after the first reward is the geodesic (BFS)
shortest path, whose length — averaged over this evaluation's mazes, goals
and teleport landings — is about 3.1 moves (Section 7 measures it). That
number is the reference line for the performance analysis there.


## 3. The agent: a GRU with a policy, a value, and a world model

The agent is a single GRU with 100 hidden units, topped by three heads:

$$
h_t = \mathrm{GRU}(x_t, h_{t-1}; \theta), \qquad h_0 \text{ learned},
$$

$$
\underbrace{[\pi_t,\; V_t]}_{\text{policy/value head}} = W_o h_t,
\qquad
\underbrace{\hat{y}_t}_{\text{world model}} =
W_2\,\mathrm{ReLU}\big(W_1 [h_t, \mathrm{onehot}(a_t)]\big).
$$

* **Policy/value head** (100 → 6): logits for the 5 actions plus a scalar
  state value $V(h_t)$.
* **World-model head** (105 → 33 → 33, ReLU): given the hidden state and the
  *taken* action, predicts (i) the next state as 16 logits and (ii) the
  reward location as 16 logits (the middle logit is unused). This head is the
  network's learned model of "what happens if I do this" and "where I believe
  the reward is". It is trained purely as an auxiliary objective — nothing
  forces the *policy* to use it.
* **Learnable initial state** $h_0$: at every episode start the hidden state
  resets to $h_0$, a free parameter. Across episodes the weights are frozen;
  all within-episode adaptation is carried by $h_t$.

In NeuralRNN this is one `ActorCriticModel` whose core is any recurrent core
(here `GatedRNNConfig(cell_type="gru", trainable_h0=True)`); the world model
is the generic *auxiliary head* (`aux_head_*` config fields with named
output slices). The same agent class covers the value-RNN of notebook 19 and
the plume agent of notebook s1 — only the config differs.


In [ ]:
# --- build the agent (this exact function is used for training) ----------
def build_agent(seed=61, hidden=100):
    torch.manual_seed(seed)
    core = GatedRNNConfig(model_type="gated_rnn", cell_type="gru",
                          input_dim=88, latent_dim=hidden, trainable_h0=True)
    cfg = ActorCriticConfig(
        core_config=core.to_dict(), action_dim=5, action_type="discrete",
        # world-model head: [h, onehot(a)] -> 33 -> 33 (relu)
        aux_head_out_dim=33, aux_head_hidden_dims=(33,),
        aux_head_activation="relu", aux_input="state_action",
        aux_slices={"next_state": (0, 16), "reward": (17, 33)})
    return ActorCriticModel(cfg)

agent_demo = build_agent()
n_params = sum(p.numel() for p in agent_demo.parameters())
print(f"parameters: {n_params:,}")
print("trainable h0:", agent_demo.core.h0.shape)

## 4. The planner: thinking as imagined trajectories

When the agent plays the *think* action, the environment runs a short
imagination loop using the agent's own world model and policy (the paper's
`planning_algorithm`, implemented in NeuralRNN as `WorldModelPlanner`):

1. **Where is the goal?** Read the world model's reward-location prediction
   at the current hidden state: $\hat{p} = \arg\max$ of the reward slice
   of $\hat{y}(h_t, a=\text{think})$.
2. **Imagine up to 8 steps.** At each imagined step $k$: sample an imagined
   action $\hat{a}_k$ from the policy restricted to the four moves; predict
   the imagined next state $\hat{s}_k = \arg\max$ of the next-state slice
   of $\hat{y}(\hat{h}_k, \hat{a}_k)$; build the observation the agent
   *would* see there (same walls, imagined location, clock advanced, no
   reward, no plan feedback) and roll the GRU forward on it:
   $\hat{h}_{k+1} = \mathrm{GRU}(\hat{x}_{k+1}, \hat{h}_k)$.
   Stop early when the imagined state reaches the imagined goal.
3. **Feed the result back.** The rollout is encoded as a 33-dim vector —
   a 4×8 one-hot of the imagined action sequence plus a "goal found" flag —
   and written into the observation of the *next* real step.

Crucially, step 2 runs under `torch.no_grad` (the paper uses
`Zygote.ignore`): planning is part of the environment's transition, not of
the agent's differentiable forward pass. And step 3 is the whole trick —
the imagined trajectory re-enters the network as *input*, so the GRU
dynamics can integrate "I imagined walking north-north-east into the goal"
into the hidden state that produces the next real action. Nothing about the
planner is task-specific in the framework: `WorldModelPlanner` needs only an
agent with `readout` / `aux_logits` / `recurrence` and an input-builder
callback supplied by the environment.

Let's watch it think. Below we run one episode with a (partially) trained or
fresh agent and mark each imagined trajectory on the maze.


In [ ]:
# --- task-specific helpers (inlined: the framework itself stays generic) --
import pickle

LARENA = 4                  # maze is LARENA x LARENA, toroidal
NSTATES = LARENA * LARENA
NMOVE = 4                   # physical move actions
THINK = 4                   # the "think" action index
PLAN_DEPTH = 8
PLAN_SLICE = slice(55, 88)  # planning-feedback slice of the observation
STATE_SLICE = (0, 16)       # aux-head slice: next-state logits
REWARD_SLICE = (17, 33)     # aux-head slice: reward-location logits


def find_checkpoint(run_dir):
    '''Trained weights: "final" if present, else the newest update checkpoint.'''
    final = os.path.join(run_dir, "final")
    if os.path.isdir(final):
        return final
    cands = [d for d in os.listdir(run_dir) if d.startswith("checkpoint")]
    cands.sort(key=lambda d: int(d.split("update")[1]))
    return os.path.join(run_dir, cands[-1])


def load_agent(run_dir):
    '''Load a trained maze agent and bind a fresh WorldModelPlanner.'''
    agent = AutoModel.from_pretrained(find_checkpoint(run_dir)).eval()
    planner = WorldModelPlanner(plan_depth=PLAN_DEPTH, n_move_actions=NMOVE,
                                state_slice=STATE_SLICE,
                                reward_slice=REWARD_SLICE, seed=0).bind(agent)
    return agent, planner


def make_eval_env(planner, seed):
    '''Single MazeEnv with the planner wired to env index 0.'''
    env = MazeEnv(seed=seed)
    env.planner, env.planner_env_index = planner, 0
    return env


@torch.no_grad()
def greedy_step(agent, obs, z, allowed=None):
    '''One greedy (argmax-policy) step on a single observation.

    Returns (action, z_new, probs, value). `allowed` restricts the argmax
    (e.g. [0,1,2,3] reproduces the paper's no-planning ablation); `probs`
    are always the *unmasked* policy probabilities.
    '''
    x = torch.as_tensor(np.asarray(obs, dtype=np.float32)[None])
    z_new = agent.recurrence(x, z)
    logits = agent.readout(z_new)[0]
    probs = torch.softmax(logits, dim=-1).numpy()
    if allowed is not None:
        mask = torch.full_like(logits, -torch.inf)
        mask[list(allowed)] = 0.0
        logits = logits + mask
    action = int(logits.argmax())
    hook = getattr(agent, "planner_hook", None)  # keep planner z-cache fresh
    if hook is not None:
        hook(z_new.detach())
    return action, z_new, probs, float(agent.value(z_new)[0])


def dist_to_goal(walls, goal):
    '''Geodesic (wall-aware) distance from every state to `goal` (BFS).'''
    dists = np.full(NSTATES, np.inf)
    dists[goal] = 0.0
    live = [goal]
    while live:
        new_live = []
        for s in live:
            x, y = divmod(s, LARENA)
            for a, (dx, dy) in enumerate(DIRS):
                if walls[s, a] > 0.5:
                    continue
                ns = LARENA * ((x + dx) % LARENA) + ((y + dy) % LARENA)
                if np.isinf(dists[ns]):
                    dists[ns] = dists[s] + 1
                    new_live.append(ns)
        live = new_live
    return dists


def mean_dist_to_goal(walls, goal):
    '''Mean geodesic distance to goal over all non-goal states.'''
    d = dist_to_goal(walls, goal)
    return float(np.mean(np.concatenate([d[:goal], d[goal + 1:]])))


def parse_trials(rews, acts):
    '''Split an episode into trials (one trial = journey between rewards).

    Returns per-timestep arrays trial_id (1-based), trial_t (iteration
    within trial) and trial_anum (physical-action number within trial;
    think actions do not increment it). Mirrors the paper's
    analyse_rollout_timing.jl, including its void-iteration convention:
    the first iteration of every trial after the first is the teleport
    iteration -- the agent is still standing on the goal, its action is
    overridden by the teleport, and planning is blocked there. That
    iteration gets trial_anum = 0 so the wasted move cannot shift the
    action numbering of the genuine steps. Trial 1 starts at episode
    onset (no teleport), so its first iteration is counted normally.
    '''
    T = len(acts)
    rew_ts = np.nonzero(rews > 0.5)[0]
    starts = np.concatenate([[0], rew_ts + 1])
    ends = np.concatenate([rew_ts, [T - 1]])
    trial_id = np.zeros(T, dtype=int)
    trial_t = np.zeros(T, dtype=int)
    trial_anum = np.zeros(T, dtype=int)
    for k, (s, e) in enumerate(zip(starts, ends), start=1):
        if s > e:
            continue
        idx = np.arange(s, e + 1)
        trial_id[idx] = k
        trial_t[idx] = idx - s + 1
        anum = 1
        for j, t in enumerate(idx):
            if k > 1 and j == 0:
                continue            # void teleport iteration: anum stays 0
            trial_anum[t] = anum
            if acts[t] < THINK:
                anum += 1
    return trial_id, trial_t, trial_anum


def moves_per_trial(rews, acts):
    '''Number of physical actions in each completed trial, excluding the
    void teleport iteration that opens every post-reward trial.'''
    rew_ts = np.nonzero(rews > 0.5)[0]
    starts = np.concatenate([[0], rew_ts + 1])
    out = []
    for k, (s, e) in enumerate(zip(starts, rew_ts), start=1):
        if s > e:
            continue
        idx = np.arange(s, e + 1)
        if k > 1:
            idx = idx[1:]           # skip the void teleport iteration
        out.append(int(np.sum(acts[idx] < THINK)))
    return out


def reward_belief_prob(agent, states, actions, goals):
    '''Softmax probability the world model assigns to the true goal.'''
    z = torch.as_tensor(states, dtype=torch.float32)
    a = torch.as_tensor(actions, dtype=torch.long)
    with torch.no_grad():
        logits = agent.aux_logits(z, a)[:, REWARD_SLICE[0]:REWARD_SLICE[1]]
        probs = torch.softmax(logits, dim=-1).numpy()
    goals = np.broadcast_to(np.asarray(goals), (len(probs),))
    return probs[np.arange(len(probs)), goals]


def get_walls(ep):
    '''(16, 4) wall array stored in the episode info.'''
    return np.asarray(ep["info"]["walls"]).reshape(NSTATES, NMOVE)


def get_goal(ep):
    return int(ep["info"]["goal_loc"])


def step_rollouts(ep):
    '''Per-step RolloutResult (or None) from the recorded step infos.'''
    return [si.get("rollout") for si in ep.get("step_infos", [])]


def cached(path, compute):
    '''Load a cached analysis result; compute and store it if missing.'''
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    out = compute()
    with open(path, "wb") as f:
        pickle.dump(out, f)
    return out


def collect_eval_episodes(agent, planner, n_episodes=200, n_envs=20, seed=1):
    '''Greedy eval episodes with full per-step info (vectorized).'''
    envs = SyncVectorEnv([
        make_env("maze", seed=seed + i) for i in range(n_envs)])
    bind_planners(envs, planner)
    return collect_episodes(agent, envs, n_episodes, deterministic=True,
                            keep_step_infos=True, progress=True)


RUN_DIR = os.path.join(MODEL_DIR, f"seed{SEED}_full")   # the merged run


def split_jumps(states):
    '''State list -> (xs, ys) with NaN where a step wraps the torus or
    teleports, so plot() breaks the line instead of drawing a segment
    straight across the maze (which would look like walking through
    walls).'''
    xs, ys = [], []
    px = py = None
    for s in states:
        x, y = divmod(int(s), LARENA)
        if px is not None and abs(x - px) + abs(y - py) > 1:
            xs.append(np.nan)
            ys.append(np.nan)
        xs.append(x)
        ys.append(y)
        px, py = x, y
    return xs, ys
print("helpers ready; trained run:", RUN_DIR)

## 5. Training (or loading) the agent

The agent is trained on-policy with the paper's actor-critic recipe,
implemented in NeuralRNN as **`A2CPredLoss`** on top of the generic
`RLTrainer`. Per update, the trainer rolls 40 copies of the environment for
170 steps each (episodes are aligned: hidden states reset at episode
boundaries, and a rollout always starts at the beginning of an episode), then
takes a single gradient step on

$$
\mathcal{L} = \underbrace{-\beta_r\,\mathbb{E}\big[\hat{A}_t \log \pi(a_t \mid h_t)\big]}_{\text{REINFORCE with baseline}}
+ \underbrace{\beta_v\,\tfrac{1}{2}\mathbb{E}\big[(V(h_t) - \hat{R}_t)^2\big]}_{\text{value regression}}
- \underbrace{\beta_e\,\mathbb{E}\big[\mathcal{H}[\pi]\big]}_{\text{entropy bonus}}
+ \underbrace{\beta_p\,\mathbb{E}\big[\mathrm{CE}(\hat{y}_t^{\text{state}}, s_{t+1}) + \mathrm{CE}(\hat{y}_t^{\text{rew}}, p^*)\big]}_{\text{world-model prediction}}
$$

with the paper's coefficients $\beta_r = 1$, $\beta_v = 0.05$,
$\beta_e = 0.05$, $\beta_p = 0.5$, Adam (lr $10^{-3}$,
$\epsilon = 10^{-8}$), undiscounted returns ($\gamma = 1$) with GAE
$\lambda = 0.95$, and no advantage normalization. Three details worth
knowing:

* **Advantages are undiscounted.** $\gamma = 1$ with GAE $\lambda = 0.95$ —
  the episode is short and every reward counts the same, whenever it arrives.
* **The entropy bonus is the paper's KL regularizer.** Jensen et al.
  regularize the policy toward a uniform prior; on the policy simplex that
  is, up to a constant, the same gradient as rewarding entropy.
* **The world model is just an auxiliary loss.** Nothing in $\mathcal{L}$
  routes through the planner: the world-model head is shaped only by the
  prediction cross-entropies, and planning runs under `no_grad`. Whatever the
  policy learns to *do* with imagined trajectories is emergent.

We trained for **200 M environment steps** in total (~29 000 updates,
≈ 3.2 M episodes, roughly 25 h on a CPU). The paper trains for 8 M episodes
(200 k batches), so this is about 40% of the full budget; Section 12
discusses what that does and does not change.

The cell below *is* the whole training pipeline — no external scripts. The
full run takes about a day, so the notebook ships with the trained weights
in `models/s2/seed61_full/` and simply reloads them; set `TRAIN = True` (or
point `RUN_DIR` at a fresh directory) to train your own from scratch.

In [ ]:
# --- train the agent, or load the finished run if it is already there -------
TRAIN = False                  # set True to train from scratch (~25 h CPU)
TOTAL_TIMESTEPS = 200_000_000  # the budget our shipped agent was trained with

if not TRAIN and os.path.isdir(os.path.join(RUN_DIR, "final")):
    agent, planner = load_agent(RUN_DIR)
    print("loaded trained agent:", find_checkpoint(RUN_DIR))
else:
    import json
    os.makedirs(RUN_DIR, exist_ok=True)
    agent = build_agent(seed=SEED)
    planner = WorldModelPlanner(plan_depth=PLAN_DEPTH, n_move_actions=NMOVE,
                                state_slice=STATE_SLICE,
                                reward_slice=REWARD_SLICE,
                                seed=SEED).bind(agent)
    train_envs = SyncVectorEnv([make_env("maze", seed=10_000 * SEED + i)
                                for i in range(40)])
    bind_planners(train_envs, planner)
    args = RLTrainingArguments(
        total_timesteps=TOTAL_TIMESTEPS, num_envs=40, num_steps=170,
        num_minibatches=1, update_epochs=1,          # strictly on-policy
        gamma=1.0, estimator="nstep", lr=1e-3, adam_eps=1e-8,
        weight_decay=0.0, max_grad_norm=1e6, adv_norm="none",
        aux_target_dim=2,                # per-step aux targets for the WM head
        reset_at_update_start=True,      # rollouts start at episode onsets
        seed=SEED, device="cpu", output_dir=RUN_DIR, save_every=250,
        progress_bar=True)
    trainer = RLTrainer(agent, train_envs, args, loss=A2CPredLoss())
    history = trainer.train()
    with open(os.path.join(RUN_DIR, "history.json"), "w") as f:
        json.dump(history["scalars"], f)
    print("trained from scratch ->", RUN_DIR)

## 6. A trained episode, step by step

Before the population statistics, watch one greedy episode up close. The
three panels below show the first three trials — trial 1 is exploration (the
goal is still hidden), trials 2 and 3 are the return trips the paper is
about.

In [ ]:
# --- one greedy episode: the first three trials, one panel each -------------
env = make_eval_env(planner, seed=2)  # picked: 11 rewards, 14 rollouts
                                        # (8 reach the goal), zero visual
                                        # wall crossings (tests/maze/scan_demo_seed.py)
obs, _ = env.reset()
z = agent.init_state(1, "cpu")
path, rollouts, acts, rews = [env.agent_state], [], [], []
done = False
while not done:
    a, z, _, _ = greedy_step(agent, obs, z)
    obs, r, done, _, si = env.step(a)
    acts.append(a)
    rews.append(r)
    path.append(si["agent_loc"])       # path[t+1] = position after iteration t
    if si.get("rollout") is not None:
        res = si["rollout"]
        rollouts.append((len(acts) - 1, si["agent_loc"],
                         res.states[res.states >= 0].tolist(),
                         bool(res.found_goal)))

n_rew = int(sum(r > 0.5 for r in rews))
n_think = int(sum(a == THINK for a in acts))
print(f"episode: {len(acts)} iterations, {n_rew} rewards, {n_think} thinks "
      f"({len(rollouts)} rollouts)")

# trial boundaries; for trials >= 2 the first iteration is the void teleport
# step (see parse_trials), so the drawn path starts right after it
rew_ts = np.nonzero(np.array(rews) > 0.5)[0]
starts = np.concatenate([[0], rew_ts + 1])

from matplotlib.lines import Line2D
N_SHOW = 3                             # one panel per trial, first 3 trials
fig, axes = plt.subplots(1, N_SHOW, figsize=(10, 3.5))
for k, ax in enumerate(axes, start=1):
    s, e = starts[k - 1], rew_ts[k - 1]
    first = s if k == 1 else s + 1     # skip the void teleport iteration
    tpath = path[first:e + 2]          # start position .. goal
    n_moves = int(np.sum(np.array(acts)[first:e + 1] < THINK))
    draw_maze(ax, env.walls, goal=env.goal, path=tpath,
              title=f"trial {k}: {n_moves} moves")
    for t, s0, seq, found in rollouts:
        if not (first <= t <= e):
            continue
        xs, ys = split_jumps([s0] + list(seq))  # break at torus wrap-arounds
        ax.plot(xs, ys, "s--", lw=1.1, ms=3, zorder=4,
                color="firebrick" if found else "lightsalmon")
handles = [
    Line2D([], [], marker="*", color="goldenrod", ls="", ms=13, label="goal"),
    Line2D([], [], color="steelblue", marker="o", ms=4, label="physical path"),
    Line2D([], [], color="firebrick", ls="--", marker="s", ms=4,
           label="imagined (reached goal)"),
    Line2D([], [], color="lightsalmon", ls="--", marker="s", ms=4,
           label="imagined (did not)"),
]
fig.legend(handles=handles, loc="center right", bbox_to_anchor=(1.05, 0.5),
           fontsize=8.5, frameon=False)
fig.tight_layout(rect=[0, 0, 0.87, 1])
fig.savefig(os.path.join(FIG_DIR, "fig_planner_episode.png"),
            bbox_inches="tight")
plt.show()

Blue is what the body did; the red dashes are what the network *imagined*
while standing still — dark red if the rollout reached the goal, light red
if not. Splitting the episode by trial (one panel each) keeps the paths
legible; even so the thinks stack on top of the walks, and the animation
below resolves them in time. Two things to look for: thinking clusters at
the start of the return trips (right after a teleport, when the way back to
the goal must be re-planned), and imagined trajectories that run along
plausible corridors and often terminate on the goal. The rest of the
notebook quantifies both impressions.

Two things you may notice in individual episodes are *not* bugs. First, the
agent occasionally never reaches the goal before the episode's time budget
runs out — about 13% of trials across our 200 evaluation episodes, almost
all of them trial 1, where the goal is still hidden and unlucky exploration
simply runs out of clock. Second, this arena is a torus: a step off one
edge legitimately re-enters on the opposite edge. We break the line at such
wrap-arounds (and at teleports) rather than drawing a segment straight
across the maze, which would look exactly like walking through a wall.

### Animation

In [ ]:
# --- the same episode as an HTML animation ----------------------------------
# One frame per sub-step: physical moves extend the blue path; a think
# action grows the imagined trajectory step by step (dark red = reached the
# goal, light red = not), and the red trace is cleared when the think ends.
# After each reward the agent teleports and the blue trace restarts, so
# every trial is drawn as its own journey.
from matplotlib import animation
from IPython.display import HTML


def episode_frames(records, start_state):
    '''Per-iteration records -> animation frames.

    records: ("move", state, got_reward) | ("think", imagined_states,
    found_goal, got_reward) | ("teleport", new_state) -- the void iteration
    after a reward, whose move is overridden by the teleport.
    Frames: (blue_states, red_states_or_None, red_found, n_rew, label).
    '''
    frames, blue, n_rew, n_trial = [], [start_state], 0, 1
    for rec in records:
        if rec[0] == "teleport":
            blue = [rec[1]]            # new trial: clear the physical trace
            n_trial += 1
            frames.append((list(blue), None, False, n_rew,
                           f"trial {n_trial} - teleport"))
        elif rec[0] == "move":
            _, state, got = rec
            blue.append(state)
            n_rew += int(got)
            frames.append((list(blue), None, False, n_rew,
                           f"trial {n_trial} - move"))
        else:
            _, seq, found, got = rec
            n_rew += int(got)
            for k in range(2, len(seq) + 1):   # the imagined path grows
                frames.append((list(blue), seq[:k], found, n_rew,
                               f"trial {n_trial} - think"))
            frames.append((list(blue), None, found, n_rew,
                           f"trial {n_trial} - think done"))
    return frames


# re-run the demo episode, keeping the rollout of every think action
env = make_eval_env(planner, seed=56)
obs, _ = env.reset()
start_state = env.agent_state
z = agent.init_state(1, "cpu")
records = []
done = False
while not done:
    a, z, _, _ = greedy_step(agent, obs, z)
    obs, r, done, _, si = env.step(a)
    res = si.get("rollout")
    if si.get("at_rew"):
        records.append(("teleport", si["agent_loc"]))
    elif a == THINK and res is not None:
        records.append(("think",
                        [si["agent_loc"]] + res.states[res.states >= 0].tolist(),
                        bool(res.found_goal), r > 0.5))
    else:
        records.append(("move", si["agent_loc"], r > 0.5))
frames = episode_frames(records, start_state)
print(f"{len(records)} iterations -> {len(frames)} frames")


def draw_walls(ax, walls):
    '''Internal walls of the 4x4 maze (each drawn once).'''
    for s in range(NSTATES):
        x, y = divmod(s, LARENA)
        for a, (dx, dy) in enumerate(DIRS):
            if walls[s, a] < 0.5:
                continue
            if abs(x - (x + dx) % LARENA) > 1 or abs(y - (y + dy) % LARENA) > 1:
                continue                     # boundary walls drawn once
            if dx == 1:
                ax.plot([x + 0.5, x + 0.5], [y - 0.5, y + 0.5], "k-", lw=2)
            elif dy == 1:
                ax.plot([x - 0.5, x + 0.5], [y + 0.5, y + 0.5], "k-", lw=2)


fig, ax = plt.subplots(figsize=(3.4, 2.5))


def draw_frame(frame):
    blue, red, found, n_rew, label = frame
    ax.clear()
    ax.set_aspect("equal")
    draw_walls(ax, env.walls)
    gx, gy = divmod(env.goal, LARENA)
    ax.scatter([gx], [gy], marker="*", s=300, color="goldenrod", zorder=5)
    if blue:
        xs, ys = split_jumps(blue)
        ax.plot(xs, ys, "o-", color="steelblue", lw=2.0, ms=4, zorder=3)
        ax.scatter([xs[-1]], [ys[-1]], marker="o", s=110, facecolor="none",
                   edgecolor="steelblue", lw=2, zorder=4)
    if red:
        xs, ys = split_jumps(red)
        ax.plot(xs, ys, "s--", lw=1.6, ms=4, zorder=4,
                color="firebrick" if found else "lightsalmon")
    ax.set_xlim(-0.6, LARENA - 0.4)
    ax.set_ylim(-0.6, LARENA - 0.4)
    ax.set_xticks(range(LARENA))
    ax.set_yticks(range(LARENA))
    ax.set_title(f"{label}  |  rewards so far: {n_rew}", fontsize=10)
    handles = [
        Line2D([], [], marker="*", color="goldenrod", ls="", ms=13,
               label="goal"),
        Line2D([], [], color="steelblue", marker="o", ms=4,
               label="physical path"),
        Line2D([], [], color="firebrick", ls="--", marker="s", ms=4,
               label="imagined rollout (reached goal)"),
        Line2D([], [], color="lightsalmon", ls="--", marker="s", ms=4,
               label="imagined rollout (did not)"),
    ]
    # ax.legend(handles=handles, loc="upper left", fontsize=6,
    #           frameon=False, bbox_to_anchor=(1.02, 1.0))


anim = animation.FuncAnimation(fig, draw_frame, frames=frames,
                               interval=125)          # 8 fps
html = anim.to_jshtml(fps=2)
with open(os.path.join(FIG_DIR, "fig_planner_episode_anim.html"), "w") as f:
    f.write(html)
plt.close(fig)
HTML(html)

## 7. Behavior of the trained agent (Fig 2)

The paper's first result is that the agent *forages* the way a foraging
animal does. We collect 200 greedy episodes on fresh mazes (cached in
`models/s2/eval/`) and ask three questions:

1. **Does it exploit what it learns within an episode?** Trial 1 is
   exploration: the goal is unknown and must be stumbled upon. From trial 2
   on, the optimal policy is the geodesic shortest path back to the
   remembered goal — on these mazes that averages **≈ 3 moves** (dashed
   line: the BFS optimum for each episode's maze, averaged). If the network
   really stores the goal in its hidden state, moves-to-goal should drop
   toward that line after the first reward.
2. **Does it think more when thinking is needed?** Humans in the paper's
   matched experiment take longer before acting in harder mazes, and so does
   the model: the "response time" of the network — iterations (i.e. thinks,
   since a physical move is one iteration) spent per physical action — grows
   with the geodesic distance from the trial-start cell to the goal. A
   complementary view (the paper's Fig 2d): *within* one return trip, which
   steps carry the thinking?
3. **Does its belief about the goal sharpen as it explores?** Reading the
   world model's reward-location head during trial 1 should show a belief
   that is near chance (1/16 ≈ 0.06) at first and grows with the number of
   distinct states visited — the signature of an agent accumulating evidence
   about a hidden variable.

In [ ]:
# --- collect (or load) 200 greedy evaluation episodes ----------------------
def _collect():
    return collect_eval_episodes(agent, planner, n_episodes=200)

eps = cached(os.path.join(EVAL_DIR, "eval_episodes.pkl"), _collect)
n_rew_ep = [int(np.sum(ep["rewards"] > 0.5)) for ep in eps]
print(f"{len(eps)} episodes; rewards per episode: "
      f"mean {np.mean(n_rew_ep):.1f}, median {np.median(n_rew_ep):.0f}, "
      f"max {np.max(n_rew_ep)}")

# --- Fig 2ab: exploitation performance and thinking time -------------------
RMIN = 4   # only episodes with >= RMIN rewards enter the by-trial curve
good = [ep for ep in eps if np.sum(ep["rewards"] > 0.5) >= RMIN]
perfs = np.array([moves_per_trial(ep["rewards"], ep["actions"])[:RMIN]
                  for ep in good])
baseline = np.mean([mean_dist_to_goal(get_walls(ep), get_goal(ep))
                    for ep in good])

# response time: iterations per physical action, by start-goal distance;
# also split by step within trial (the paper's Fig 2d), in ms at the paper's
# assumed 120 ms per rollout. Action numbering follows the paper
# (analyse_rollout_timing.jl): the void teleport iteration that opens every
# post-reward trial carries action number 0 and drops out of the counts, and
# the trial-start distance is read at the post-teleport location -- our
# step_infos entry at that void iteration -- not after the first real move.
ROLLOUT_MS = 120.0
by_dist = {}
by_step = {}
for ep in eps:
    rews, acts = ep["rewards"], ep["actions"]
    trial_id, _, trial_anum = parse_trials(rews, acts)
    walls, goal = get_walls(ep), get_goal(ep)
    dists = dist_to_goal(walls, goal)
    locs = np.array([si["agent_loc"] for si in ep["step_infos"]])
    n_rew = int(np.sum(rews > 0.5))
    for trial in range(2, min(15, n_rew + 1)):
        inds = np.nonzero(trial_id == trial)[0]
        if len(inds) == 0:
            continue
        anums = trial_anum[inds]         # anum 0 = void teleport iteration
        d = int(dists[locs[inds[0]]])    # post-teleport (trial-start) loc
        entry = by_dist.setdefault(d, [])
        for anum in range(1, anums[-1] + 1):
            n_iter = int(np.sum(anums == anum))
            entry.append(n_iter)
            by_step.setdefault((d, anum), []).append((n_iter - 1) * ROLLOUT_MS)

ds = sorted(by_dist)
rt_mean = [np.mean(by_dist[d]) for d in ds]
rt_sem = [np.std(by_dist[d]) / np.sqrt(len(by_dist[d])) for d in ds]

fig, axes = plt.subplots(1, 3, figsize=(13.8, 3.6))
ax = axes[0]
x = np.arange(1, RMIN + 1)
ax.errorbar(x, perfs.mean(axis=0),
            yerr=perfs.std(axis=0) / np.sqrt(len(perfs)),
            fmt="o-", color="steelblue", lw=1.8, ms=5, capsize=3,
            label=f"agent (n={len(perfs)} episodes)")
ax.axhline(baseline, color="k", ls="--", lw=1.2,
           label=f"optimal (BFS) = {baseline:.2f}")
ax.set_xlabel("trial within episode")
ax.set_ylabel("moves to goal")
ax.set_xticks(x)
ax.legend(fontsize=8, frameon=False)
ax.set_title("exploitation approaches the optimum")

ax = axes[1]
ax.errorbar(ds, rt_mean, yerr=rt_sem, fmt="s-", color="firebrick",
            lw=1.8, ms=5, capsize=3)
ax.set_xlabel("geodesic distance to goal at trial start")
ax.set_ylabel("iterations per action")
ax.set_title("thinking time grows with difficulty")

ax = axes[2]
cols = {2: "tab:blue", 3: "tab:orange", 4: "tab:green", 5: "tab:red"}
for d in (2, 3, 4, 5):
    ts = list(range(1, d + 1))
    mu = [np.mean(by_step.get((d, t), [np.nan])) for t in ts]
    se = [np.std(by_step.get((d, t), [0])) /
          np.sqrt(max(1, len(by_step.get((d, t), [])))) for t in ts]
    ax.plot(ts, mu, "o-", color=cols[d], lw=1.8, ms=4,
            label=f"goal dist = {d}")
    ax.fill_between(ts, np.array(mu) - se, np.array(mu) + se,
                    color=cols[d], alpha=0.2, lw=0)
ax.set_xlabel("step within trial")
ax.set_ylabel("thinking time (ms)")
ax.set_title("thinking is front-loaded in the return trip")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig2_behavior.png"))
plt.show()

print(f"moves per trial 1..{RMIN}: {perfs.mean(axis=0).round(2)} "
      f"(BFS optimum {baseline:.2f})")
print("iterations/action by distance: "
      f"{ {d: round(m, 2) for d, m in zip(ds, rt_mean)} }")
for d in (2, 3, 4, 5):
    print(f"thinking time (ms), goal dist {d}: "
          f"{[round(np.mean(by_step.get((d, t), [np.nan]))) for t in range(1, d + 1)]}")

**Reading the panels.** Left: trial 1 is expensive — the agent must
physically search for an unknown goal (~9 moves) — and from trial 2 on the
curve drops to ~4 moves, within about one move of the geodesic optimum
(dashed). The ordering, not the gap, is the qualitative result: the goal
location is genuinely carried in the recurrent state from trial 2 on. The
residual gap to the optimum is where our smaller training budget shows
(Section 12); the paper's longer-trained agent sits closer to the line.
Middle: the network spends more iterations per action (extra thinks) as the
geodesic distance from trial start to goal grows from 0 to about 5 — the
regime that covers nearly all trials; beyond that the curve gets noisy
because long geodesics are rare on a 4×4 torus. This mirrors the paper's
human result — response times scale with maze difficulty — and it is an
*emergent* allocation of a costly resource: nothing in the loss rewards
thinking in hard mazes, thinking is simply less useful when the goal is one
step away. Right (the paper's Fig 2d): within one return trip, thinking is
front-loaded — the first step after the teleport carries most of it (up to
~250 ms for the farthest goals, at the paper's assumed 120 ms per rollout)
and later steps, closer to the goal, need progressively less — and harder
trips think more at *every* step, not just the first. Action numbering
follows the paper: the teleport iteration itself (the agent's wasted action
while still standing on the goal) is not counted as a step.

In [ ]:
# --- Fig 2cd: trial-1 exploration and the sharpening goal belief ---------
rt_by_anum, unique_by_anum, dec_by_unique = {}, {}, {}
for ep in eps:
    rews, acts = ep["rewards"], ep["actions"]
    trial_id, _, trial_anum = parse_trials(rews, acts)
    inds = np.nonzero(trial_id == 1)[0]
    if len(inds) == 0:
        continue
    locs = np.array([si["agent_loc"] for si in ep["step_infos"]])
    goal = get_goal(ep)
    visited = np.zeros(NSTATES, dtype=bool)
    seen = set()
    for t in inds:
        anum = trial_anum[t]
        if anum in seen:
            continue
        seen.add(anum)
        visited[locs[t]] = True
        unique_by_anum.setdefault(anum, []).append(int(visited.sum()))
    # world-model goal belief at every trial-1 iteration
    p_goal = reward_belief_prob(agent, ep["states"], ep["actions"], goal)
    visited = np.zeros(NSTATES, dtype=bool)
    for t in inds:
        visited[locs[t]] = True
        dec_by_unique.setdefault(int(visited.sum()), []).append(
            float(p_goal[t]))

anums = sorted(unique_by_anum)
u_mean = [np.mean(unique_by_anum[a]) for a in anums]
u_sem = [np.std(unique_by_anum[a]) / np.sqrt(len(unique_by_anum[a]))
         for a in anums]
us = sorted(dec_by_unique)
d_mean = [np.mean(dec_by_unique[u]) for u in us]
d_sem = [np.std(dec_by_unique[u]) / np.sqrt(len(dec_by_unique[u]))
         for u in us]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax = axes[0]
ax.errorbar(anums, u_mean, yerr=u_sem, fmt="-", color="steelblue",
            lw=1.8, ecolor="lightsteelblue", capsize=2)
ax.plot(anums, np.minimum(anums, NSTATES), "k--", lw=1,
        label="one new state per move (max)")
ax.set_xlabel("action number (trial 1)")
ax.set_ylabel("unique states visited")
ax.set_title("trial-1 exploration covers the maze")
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
ax.errorbar(us, d_mean, yerr=d_sem, fmt="o-", color="darkorange",
            lw=1.8, ms=4, capsize=3)
ax.axhline(1 / NSTATES, color="k", ls="--", lw=1, label="chance (1/16)")
ax.set_xlabel("unique states visited so far")
ax.set_ylabel("P(world model believes true goal)")
ax.set_title("goal belief sharpens with exploration")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig2_exploration.png"))
plt.show()

print("unique states by action number: "
      f"{[round(m, 2) for m in u_mean[:10]]} ...")
print("goal-belief prob by #unique states: "
      f"{ {u: round(m, 3) for u, m in zip(us, d_mean)} }")

During trial 1 the agent covers ground efficiently (left:
unique states visited tracks the one-new-state-per-move ceiling for the
first ~8 actions — it is exploring, not pacing in circles), and its world
model's belief about the hidden goal climbs steadily above chance as the
visited set grows (right: from the 1/16 baseline to roughly double that by
the time a dozen states have been seen). The belief cannot reach high
values from exploration alone — before the first reward the goal is
*hidden*, and the agent can at most rule out where it is not. (The noisy
high end of the curve is based on few samples and includes steps right at
reward collection, where the belief is about to become certain anyway.)
The sharp jump comes when the first reward is collected; after that the
reward belief is essentially perfect, which Section 11 measures directly.


## 8. Rollouts causally improve the policy (Fig 3)

So far everything is correlational: the agent thinks, and it performs well.
The paper's second figure establishes that imagined trajectories *cause*
better behavior, in three experiments that we repeat verbatim:

**A. Prevent planning.** Mask the think logit with $-\infty$ (the paper's
`no_planning` condition) and re-run the same greedy evaluation. If rollouts
do useful work, total reward per episode should drop.

**B. Force planning.** Immediately after the first reward, *force* $n$
consecutive think actions before letting the agent move (it normally chooses
when to think on its own). If each rollout adds useful information about the
way back to the goal, moves-to-goal on trial 2 should fall with $n$. A
control condition zeroes the plan-feedback input during the movement phase,
to check that any improvement travels through the feedback channel rather
than through, say, extra compute time.

**C. One rollout, one bit of evidence.** At the first think of trial 2,
resample the rollout (the planner samples imagined actions from the policy,
so repeated calls differ) until it either reaches the imagined goal or
doesn't, and measure the policy and value *at the first forward pass that
sees the rollout feedback*. The paper's signature result: a goal-**finding**
rollout boosts the policy probability of its own first imagined action and
raises the value estimate; a goal-**missing** rollout does the opposite. The
control is the same measurement with no rollout at all.

All three experiments are causal interventions on a frozen network — no
retraining. Results are cached in `models/s2/eval/fig3.pkl`.


In [ ]:
# --- A. planning ablation --------------------------------------------------
def run_ablation(n_episodes=100, seed0=100):
    '''Total reward per episode with the think action allowed vs masked.'''
    out = {}
    for allow in (True, False):
        totals = []
        for i in range(n_episodes):
            env = make_eval_env(planner, seed0 + i)
            obs, _ = env.reset()
            z = agent.init_state(1, "cpu")
            done, total = False, 0.0
            while not done:
                a, z, _, _ = greedy_step(agent, obs, z,
                                         allowed=[0, 1, 2, 3, 4] if allow
                                         else [0, 1, 2, 3])
                obs, r, done, _, _ = env.step(a)
                total += r
            totals.append(total)
        out["plan" if allow else "no_plan"] = np.array(totals)
    return out


# --- B. forced rollouts after the first reward -----------------------------
def run_forced(n_reps=50, nplans=range(0, 8), zero_plan=False, seed0=0,
               max_iters=100):
    '''Moves to the second reward after `nplan` enforced think actions.

    Every rep replays the *same* maze (same seed) for each value of nplan,
    so the comparison across nplan is within-maze. Returns (dts, mindists):
    moves to goal, and the BFS distance at the moment movement begins.
    '''
    nplans = list(nplans)
    dts = np.full((n_reps, len(nplans)), np.nan)
    mindists = np.full((n_reps, len(nplans)), np.nan)
    for rep in range(n_reps):
        dists = None
        for ip, nplan in enumerate(nplans):
            env = make_eval_env(planner, seed0 + rep)
            obs, _ = env.reset(seed=seed0 + rep)
            if dists is None:
                dists = dist_to_goal(env.walls, env.goal)
            z = agent.init_state(1, "cpu")
            exploit, to_go, nact, t, done = False, nplan, 0, 0, False
            void = False                   # next iteration is the void
            while not done and t < max_iters:   # teleport step (parse_trials)
                t += 1
                obs_in = obs.copy()
                if zero_plan and exploit:
                    obs_in[PLAN_SLICE] = 0.0     # feedback-channel control
                a, z, probs, _ = greedy_step(agent, obs_in, z)
                if exploit and void:
                    # agent still on the goal: the move is overridden by the
                    # teleport and planning is blocked -- do not count it and
                    # do not spend an enforced rollout on it
                    a = int(np.argmax(probs[:4]))
                    void = False
                elif exploit and to_go > 0:
                    a, to_go = THINK, to_go - 1  # enforce a rollout
                    if to_go == 0:
                        mindists[rep, ip] = dists[env.agent_state]
                elif exploit:
                    a = int(np.argmax(probs[:4]))
                    nact += 1
                obs, r, done, _, _ = env.step(a)
                if r > 0.5:
                    if exploit:
                        dts[rep, ip] = nact      # second reward: trial over
                        break
                    exploit = True               # first reward
                    void = True
    return dts, mindists


# --- C. resample one rollout until it finds / misses the goal --------------
def run_causal(n_reps=50, seed0=0, max_resample=100):
    '''Policy/value change after a goal-finding vs goal-missing rollout.

    Conditions: True = forced to find the goal, False = forced to miss,
    None = no-rollout control. "after" = the first forward pass that sees
    the rollout feedback (the iteration right after the think).
    '''
    keys = ["p_sim_new", "p_sim_old", "V_new", "V_old"]
    data = {k: np.full((3, n_reps), np.nan) for k in keys}
    for rep in range(n_reps):
        for icond, cond in enumerate([True, False, None]):
            env = make_eval_env(planner, seed0 + rep)
            obs, _ = env.reset(seed=seed0 + rep)
            builder = env._make_input_builder()
            z = agent.init_state(1, "cpu")
            exploit = stored = just_planned = False
            sim_a, probs_old, done = -1, None, False
            while not done:
                a, z, probs, value = greedy_step(agent, obs, z)
                if just_planned and not stored:
                    # this forward pass saw the rollout feedback
                    data["p_sim_new"][icond, rep] = probs[sim_a]
                    data["p_sim_old"][icond, rep] = probs_old[sim_a]
                    data["V_new"][icond, rep] = value
                    stored, just_planned = True, False
                if stored and a == THINK:
                    a = int(np.argmax(probs[:4]))  # one plan per trial 2
                probs_old = probs
                at_rew = env.agent_state == env.goal
                if exploit and not stored and a == THINK and not at_rew:
                    if cond is None:               # no-rollout control
                        sim_a = int(np.argmax(probs[:4]))
                        a = sim_a                  # take the move instead
                        data["V_old"][icond, rep] = value
                        just_planned = True
                    else:
                        ntry = 0
                        while True:                # resample until cond met
                            ntry += 1
                            res = planner.rollout(z, THINK, builder)
                            if res.found_goal == cond or ntry > max_resample:
                                break
                        valid = res.actions[res.actions >= 0]
                        sim_a = int(valid[0]) if len(valid) else -1
                        data["V_old"][icond, rep] = value
                        # apply the rollout by hand: a think step costs
                        # planning_time, does not move, and writes the plan
                        # feedback into the next observation
                        obs = env._obs(THINK, 0.0, res.plan_input)
                        env.time += env.planning_time
                        done = not (env.time < env.budget + 1.0 - 1e-2)
                        just_planned = True
                        continue
                obs, r, done, _, _ = env.step(a)
                if r > 0.5:
                    if exploit and stored:
                        break                    # second reward: done
                    exploit = True
    return data


def _fig3():
    abl = run_ablation()
    dts, mindists = run_forced()
    dts_z, _ = run_forced(zero_plan=True, seed0=0)
    causal = run_causal()
    return {"ablation": abl, "dts": dts, "mindists": mindists,
            "dts_zero": dts_z, "causal": causal}


F3 = cached(os.path.join(EVAL_DIR, "fig3.pkl"), _fig3)
abl, dts, dts_z, causal = (F3["ablation"], F3["dts"], F3["dts_zero"],
                           F3["causal"])
print(f"[A] reward per episode: planning {abl['plan'].mean():.2f} +- "
      f"{abl['plan'].std() / np.sqrt(len(abl['plan'])):.2f} | "
      f"no-planning {abl['no_plan'].mean():.2f} +- "
      f"{abl['no_plan'].std() / np.sqrt(len(abl['no_plan'])):.2f}")
print(f"[B] moves to goal vs enforced rollouts: "
      f"{np.nanmean(dts, axis=0).round(2)}")
print(f"    (feedback zeroed during movement: "
      f"{np.nanmean(dts_z, axis=0).round(2)})")
for icond, name in enumerate(["goal found", "goal missed", "no rollout"]):
    dp = np.nanmean(causal["p_sim_new"][icond] - causal["p_sim_old"][icond])
    dv = np.nanmean(causal["V_new"][icond] - causal["V_old"][icond])
    print(f"[C] {name:11s}: p(first imagined action) "
          f"{np.nanmean(causal['p_sim_old'][icond]):.3f} -> "
          f"{np.nanmean(causal['p_sim_new'][icond]):.3f} (d={dp:+.3f}), "
          f"dV={dv:+.3f}")

In [ ]:
# --- Fig 3 figure ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))

ax = axes[0]
means = [abl["plan"].mean(), abl["no_plan"].mean()]
sems = [abl[k].std() / np.sqrt(len(abl[k])) for k in ("plan", "no_plan")]
ax.bar([0, 1], means, yerr=sems, width=0.55, capsize=4,
       color=["steelblue", "lightsteelblue"], edgecolor="none")
rng = np.random.default_rng(0)
for i, k in enumerate(("plan", "no_plan")):
    ax.scatter(rng.normal(i, 0.05, len(abl[k])), abl[k], s=4,
               color="k", alpha=0.25, zorder=3)
ax.set_xticks([0, 1], ["planning\nallowed", "think action\nmasked"])
ax.set_ylabel("reward per episode")
ax.set_title("A. preventing planning hurts")

ax = axes[1]
nplans = np.arange(dts.shape[1])
for arr, color, label in [(dts, "steelblue", "plan feedback intact"),
                          (dts_z, "gray", "feedback zeroed (control)")]:
    m = np.nanmean(arr, axis=0)
    s = np.nanstd(arr, axis=0) / np.sqrt(np.sum(~np.isnan(arr), axis=0))
    ax.errorbar(nplans, m, yerr=s, fmt="o-", color=color, lw=1.8, ms=4,
                capsize=3, label=label)
ax.set_xlabel("enforced rollouts after first reward")
ax.set_ylabel("moves to goal (trial 2)")
ax.set_title("B. forcing extra rollouts helps")
ax.legend(fontsize=8, frameon=False)

ax = axes[2]
names = ["goal\nfound", "goal\nmissed", "no rollout\n(control)"]
old = [np.nanmean(causal["p_sim_old"][i]) for i in range(3)]
new = [np.nanmean(causal["p_sim_new"][i]) for i in range(3)]
x = np.arange(3)
ax.bar(x - 0.18, old, width=0.36, color="lightgray",
       label="before feedback")
ax.bar(x + 0.18, new, width=0.36,
       color=["seagreen", "indianred", "gray"], label="after feedback")
for i in range(3):
    ax.annotate("", xy=(x[i] + 0.18, new[i]), xytext=(x[i] - 0.18, old[i]),
                arrowprops=dict(arrowstyle="->", color="k", lw=0.8))
ax.set_xticks(x, names)
ax.set_ylabel("P(first imagined action)")
ax.set_title("C. one rollout shifts the policy")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig3_causal.png"))
plt.show()

**Reading the panels.** (A) Masking the think action costs about 0.7 of a
reward per episode on average (6.31 → 5.63, an 11% drop). The effect is
clear, though still smaller than the paper's — their longer-trained agents
lean on rollouts more heavily and lose more. The cleaner evidence is in the
interventional panels. (B) Each enforced rollout shortens the walk back to
the goal (~5.7 → ~3.6 moves over eight rollouts), and most of the benefit
vanishes when the feedback channel is zeroed during the movement phase
(gray: ~5.4 → ~4.9) — so it is the *content* of the imagined trajectories,
delivered as input, that helps, not the pause itself. (C) is the sharpest
result: a single goal-finding rollout more than doubles the policy
probability of its own first action (0.22 → 0.48) and raises the value
estimate ($\Delta V \approx +0.3$), while a goal-missing rollout slightly
lowers that probability (0.14 → 0.13) and *cuts* value
($\Delta V \approx -0.6$). The no-rollout control drifts the other way:
without confirmatory imagination, the policy's commitment to its current
best action *decays* (0.25 → 0.12, $\Delta V \approx 0$). The network has
learned to treat its own imagination as evidence — the computational role
the paper proposes for hippocampal replay.

## 9. What replays contain (Fig 4)

If the imagined trajectories are the network's version of hippocampal
replay, they should have the content properties that make replay useful for
planning — properties nobody trained into them, since the world-model loss
only supervises *single-step* predictions. Using the same 200 evaluation
episodes, we measure four signatures from the paper:

1. **Wall conformity.** Imagined transitions should respect walls — even
   walls the agent has never physically bumped into. We score, for every
   imagined transition, whether it crosses a wall (rate to be compared with
   a chance baseline: the fraction of blocked directions at the same
   states).
2. **Goal over-representation.** The goal cell should appear in rollouts
   more often than a generic cell — replay is not a uniform sampling of the
   maze but is pulled toward what matters.
3. **Build-up over consecutive rollouts.** When the agent thinks several
   times in a row, later rollouts should find the goal more often than
   earlier ones — as if each replay sharpens the next.
4. **Behavioral read-out.** The agent's next physical action should match
   the first imagined action specifically when the rollout found the goal.


In [ ]:
# --- Fig 4: content statistics of imagined trajectories --------------------
DXY = {0: (1, 0), 1: (-1, 0), 2: (0, 1), 3: (0, -1)}


def action_between(s1, s2):
    '''Action taking s1 to s2 on the torus, or -1 if not adjacent.'''
    x1, y1 = divmod(s1, LARENA)
    x2, y2 = divmod(s2, LARENA)
    for a, (dx, dy) in DXY.items():
        if (x1 + dx) % LARENA == x2 and (y1 + dy) % LARENA == y2:
            return a
    return -1


def _fig4():
    # 1. wall conformity: did each imagined transition cross a wall?
    wall_hit, wall_chance = [], []
    # 2. goal over-representation: rollout visits true goal vs other cells
    goal_vis, ctrl_vis = [], []
    # 3. success by index within runs of consecutive rollouts
    seqs, cur = [], []
    # 4. next physical action == first imagined action, by rollout success
    follow_hit, follow_miss = [], []
    for ep in eps:
        walls, goal = get_walls(ep), get_goal(ep)
        locs = [si["agent_loc"] for si in ep["step_infos"]]
        acts = ep["actions"]
        # first physical action at or after each step (for signature 4)
        next_a, nxt = np.zeros(len(acts), dtype=int) - 1, -1
        for t in range(len(acts) - 1, -1, -1):
            if acts[t] < THINK:
                nxt = acts[t]
            next_a[t] = nxt
        cur = []
        for t, res in enumerate(step_rollouts(ep)):
            if res is None:
                if cur:
                    seqs.append(cur)
                    cur = []
                continue
            valid = res.states >= 0
            states = np.concatenate([[locs[t]], res.states[valid]])
            racts = res.actions[valid]
            for i in range(len(racts)):
                s1, s2 = int(states[i]), int(states[i + 1])
                if s1 == s2 or action_between(s1, s2) < 0:
                    continue
                wall_hit.append(walls[s1, int(racts[i])])
                wall_chance.append(walls[s1].mean())
            visited = res.states[valid]
            goal_vis.append(float(np.any(visited == goal)))
            ctrl_vis.append(float(np.mean(
                [np.any(visited == loc) for loc in range(NSTATES)
                 if loc != goal])))
            cur.append(float(np.any(visited == goal)))
            if next_a[t] >= 0 and len(racts):
                match = float(racts[0] == next_a[t])
                (follow_hit if np.any(visited == goal)
                 else follow_miss).append(match)
        if cur:
            seqs.append(cur)
    seq2 = np.array([s[:2] for s in seqs if 2 <= len(s) <= 5])
    seq3 = np.array([s[:3] for s in seqs if 3 <= len(s) <= 5])
    return {"wall_hit": np.array(wall_hit),
            "wall_chance": np.array(wall_chance),
            "goal_vis": np.array(goal_vis), "ctrl_vis": np.array(ctrl_vis),
            "seq2": seq2, "seq3": seq3,
            "follow_hit": np.array(follow_hit),
            "follow_miss": np.array(follow_miss)}


F4 = cached(os.path.join(EVAL_DIR, "fig4.pkl"), _fig4)

fig, axes = plt.subplots(1, 4, figsize=(14.5, 3.4))
ax = axes[0]
m = [F4["wall_hit"].mean(), F4["wall_chance"].mean()]
ax.bar([0, 1], m, width=0.55,
       color=["steelblue", "lightgray"])
ax.set_xticks([0, 1], ["imagined\ntransitions", "chance\nbaseline"])
ax.set_ylabel("P(cross a wall)")
ax.set_title("1. walls are respected")
for i, v in enumerate(m):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

ax = axes[1]
m = [F4["goal_vis"].mean(), F4["ctrl_vis"].mean()]
ax.bar([0, 1], m, width=0.55, color=["goldenrod", "lightgray"])
ax.set_xticks([0, 1], ["true goal", "other cells\n(mean)"])
ax.set_ylabel("P(cell appears in rollout)")
ax.set_title("2. goals are over-represented")
for i, v in enumerate(m):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

ax = axes[2]
for arr, color, lab in [(F4["seq2"], "steelblue", "runs of >= 2"),
                        (F4["seq3"], "darkorange", "runs of >= 3")]:
    if len(arr):
        x = np.arange(1, arr.shape[1] + 1)
        ax.errorbar(x, arr.mean(axis=0),
                    yerr=arr.std(axis=0) / np.sqrt(len(arr)),
                    fmt="o-", color=color, lw=1.8, ms=4, capsize=3,
                    label=f"{lab} (n={len(arr)})")
ax.set_xlabel("rollout number within a run")
ax.set_xticks([1, 2, 3])
ax.set_ylabel("P(rollout finds goal)")
ax.set_title("3. success builds up over rollouts")
ax.legend(fontsize=8, frameon=False)

ax = axes[3]
m = [F4["follow_hit"].mean(), F4["follow_miss"].mean()]
ax.bar([0, 1], m, width=0.55, color=["seagreen", "indianred"])
ax.set_xticks([0, 1], ["rollout found\ngoal", "rollout\nmissed"])
ax.set_ylabel("P(follow first imagined action)")
ax.set_title("4. behavior reads out replay")
for i, v in enumerate(m):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig4_replay_content.png"))
plt.show()

print(f"wall conformity: {F4['wall_hit'].mean():.3f} crossed vs "
      f"{F4['wall_chance'].mean():.3f} at chance "
      f"({len(F4['wall_hit'])} transitions)")
print(f"goal over-representation: {F4['goal_vis'].mean():.3f} vs "
      f"{F4['ctrl_vis'].mean():.3f} ({len(F4['goal_vis'])} rollouts)")
print(f"build-up (runs >= 3): {F4['seq3'].mean(axis=0).round(3)}")
print(f"follow first imagined action: found {F4['follow_hit'].mean():.3f} "
      f"vs missed {F4['follow_miss'].mean():.3f}")

All four signatures are present. Imagined transitions
essentially never cross walls (panel 1: 0.16% — and the chance bar shows
this is not trivial, since at the same states a direction picked at random
is blocked two times out of five), while rollouts visit the goal cell almost
twice as often as a generic cell (panel 2: 0.46 vs 0.25; the goal is one
cell out of sixteen, and most rollouts are short). When the agent strings
several thoughts together, later rollouts find the goal more often (panel 3:
~0.32 on the first rollout of a pair, rising to ~0.44 on the second; across
triples ~0.26 → 0.30 → 0.36) — evidence that consecutive replays refine a
shared computation rather than restarting it. And behavior reads the whole
thing out selectively: the agent follows its first imagined action about
69% of the time when the rollout found the goal, but only ~25% when it
didn't (panel 4). That selectivity is what keeps a *wrong* imagination from
being dangerous.

## 10. Why rollouts work: alignment with the policy gradient (Fig 5)

The paper's deepest claim is about mechanism. A rollout writes a 33-dim
feedback vector into the next observation; that input pushes the hidden
state from $h_t$ to $h_{t+1}$; the new hidden state produces a new policy.
*Which* direction in hidden-state space does the push point along? The
paper's answer: along the direction a policy-gradient step for the imagined
action would have taken. Replay, on this view, is the network doing one
step of policy improvement on itself, inside an episode, without any
gradient being computed.

Making that measurable takes three ingredients (ported from the paper's
`rollout_as_pg.jl`):

* **$R_\tau$, the reward-relevant feedback direction.** Regress minus the
  time to the next reward onto the feedback vectors across episodes: the
  combination of feedback dimensions that predicts imminent reward. A
  "useful" rollout is one whose feedback has a large component along
  $R_\tau$.
* **$\alpha_{\text{RNN}}$, what the feedback does to the hidden state.**
  The Jacobian of the GRU update with respect to the input, contracted with
  a perturbation along $R_\tau$ in the feedback channels:
  $\alpha_{\text{RNN}} = \frac{\partial\,\mathrm{GRU}(x, h)}{\partial x}\,\tilde{R}_\tau$.
* **$\alpha_{\text{PG}}$, what policy improvement would do.** The gradient
  of the (move-only, renormalized) log-policy for the first imagined action
  with respect to the hidden state:
  $\alpha_{\text{PG}} = \frac{\partial \log \pi(\hat{a}_1 \mid h)}{\partial h}$.
  For the *second* imagined action, the policy is evaluated one imagined
  move later, through the recurrent dynamics.

If replay implements policy improvement, $\alpha_{\text{RNN}}$ and
$\alpha_{\text{PG}}$ should align. The control permutes the first imagined
action in the feedback vector — same magnitude, wrong content — which should
destroy the alignment. Following the paper, cosines are computed in the
3-PC space of the $\alpha_{\text{PG}}$ vectors.


In [ ]:
# --- Fig 5: rollout feedback as an on-network policy-gradient step --------
from sklearn.decomposition import PCA

OBS_DIM = 88


def move_nocheck(state, a):
    '''Toroidal move ignoring walls (the Julia update_agent_state).'''
    x, y = divmod(state, LARENA)
    dx, dy = DXY[a]
    return LARENA * ((x + dx) % LARENA) + ((y + dy) % LARENA)


def estimate_r_tau(n_episodes=50, seed0=10_000):
    '''Regress -(time to next reward) on the plan-feedback vectors.'''
    X, y = [], []
    for i in range(n_episodes):
        env = make_eval_env(planner, seed0 + i)
        obs, _ = env.reset()
        z = agent.init_state(1, "cpu")
        done = False
        obs_list, times, rew_steps = [], [], []
        while not done:
            a, z, _, _ = greedy_step(agent, obs, z)
            obs, r, done, _, _ = env.step(a)
            obs_list.append(obs.copy())
            times.append(env.time)
            if r > 0.5:
                rew_steps.append(len(times) - 1)
        if len(rew_steps) < 2:
            continue
        # between the first action of trial 2 and the last reward
        for k in range(rew_steps[0] + 2, rew_steps[-1] + 1):
            pin = obs_list[k][PLAN_SLICE]
            if pin.sum() < 0.5:
                continue
            dts = [times[rw] - times[k] for rw in rew_steps
                   if times[rw] - times[k] >= -1e-9]
            if dts:
                X.append(pin)
                y.append(-min(dts))
    X, y = np.asarray(X), np.asarray(y)
    if len(X) < 40:
        raise RuntimeError(f"only {len(X)} usable feedback samples")
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    beta[:4] -= beta[:4].mean()   # remove the always-on first-action DC mode
    return beta / np.linalg.norm(beta)


def _flat(h):
    return np.asarray(h, dtype=np.float32).reshape(-1)


def _jac_recurrence(h, inp):
    '''(M, OBS_DIM) Jacobian of recurrence(inp, h) w.r.t. inp.'''
    h_t = torch.as_tensor(_flat(h)).detach()
    x0 = torch.as_tensor(np.asarray(inp, dtype=np.float32)).detach()

    def f(x):
        return agent.recurrence(x[None], h_t[None])[0]

    return torch.autograd.functional.jacobian(f, x0, vectorize=True).numpy()


def _policy_grads(h):
    '''(M, 4) grads of the move-only normalized log-policy w.r.t. h.'''
    h_t = torch.as_tensor(_flat(h)).detach().requires_grad_(True)
    logits = agent.readout(h_t[None])[0, :NMOVE]
    logp = logits - torch.logsumexp(logits, dim=0)
    grads = [torch.autograd.grad(logp[a], h_t, retain_graph=True)[0]
             for a in range(NMOVE)]
    return torch.stack(grads, dim=1).numpy()


def _policy_grads_through_move(h, nextinp):
    '''(M, 4) grads of the next-step log-policy w.r.t. h, through the
    recurrent update with imagined input `nextinp`.'''
    h_t = torch.as_tensor(_flat(h)).detach().requires_grad_(True)
    x = torch.as_tensor(np.asarray(nextinp, dtype=np.float32))
    h_new = agent.recurrence(x[None], h_t[None])
    logits = agent.readout(h_new)[0, :NMOVE]
    logp = logits - torch.logsumexp(logits, dim=0)
    grads = [torch.autograd.grad(logp[a], h_t, retain_graph=True)[0]
             for a in range(NMOVE)]
    return torch.stack(grads, dim=1).numpy()


def pg_alignment(r_tau, n_episodes=50, seed0=20_000):
    '''alpha_RNN / alpha_PG at exploitation-phase rollouts (+ controls).'''
    pert = np.zeros(OBS_DIM, dtype=np.float32)
    pert[PLAN_SLICE] = r_tau
    res = {k: [] for k in ["sim_as", "sim_a2s", "jacs", "jacs_shift",
                           "jacs_shift2", "gs", "gs2"]}
    for i in range(n_episodes):
        env = make_eval_env(planner, seed0 + i)
        obs, _ = env.reset()
        z = agent.init_state(1, "cpu")
        done = exploit = prev_planned = False
        prev_z = None
        while not done:
            if prev_planned:
                # obs carries the previous step's rollout feedback
                blocks = obs[PLAN_SLICE][:4 * PLAN_DEPTH] \
                    .reshape(PLAN_DEPTH, 4)
                sim_a = int(blocks[0].argmax())
                sim_a2 = int(blocks[1].argmax()) if blocks[1].sum() > 0.5 \
                    else -1
                jac = _jac_recurrence(prev_z, obs) @ pert
                # control: permute the first rollout action in the feedback
                inp1 = obs.copy()
                b1 = inp1[PLAN_SLICE][:4 * PLAN_DEPTH].reshape(PLAN_DEPTH, 4)
                b1[0, sim_a], b1[0, (sim_a + 1) % 4] = 0.0, 1.0
                res["sim_as"].append(sim_a)
                res["sim_a2s"].append(sim_a2)
                res["jacs"].append(jac)
                res["jacs_shift"].append(_jac_recurrence(prev_z, inp1) @ pert)
                res["gs"].append(_policy_grads(prev_z))
                if sim_a2 >= 0:
                    inp2 = obs.copy()
                    b2 = inp2[PLAN_SLICE][:4 * PLAN_DEPTH] \
                        .reshape(PLAN_DEPTH, 4)
                    b2[1, sim_a2], b2[1, (sim_a2 + 1) % 4] = 0.0, 1.0
                    res["jacs_shift2"].append(
                        _jac_recurrence(prev_z, inp2) @ pert)
                    # imagined next input, as if a_1 had been executed
                    nextinp = obs.copy()
                    nextinp[PLAN_SLICE] = 0.0
                    # NOTE: the Julia script adds (1 - 0.3) raw to the
                    # time/50 channel (likely an oversight); ported as-is
                    nextinp[6] += (1.0 - 0.3)
                    nextinp[:5] = 0.0
                    nextinp[sim_a] = 1.0
                    nextinp[7:23] = 0.0
                    nextinp[7 + move_nocheck(env.agent_state, sim_a)] = 1.0
                    res["gs2"].append(
                        _policy_grads_through_move(prev_z, nextinp))
            a, z, _, _ = greedy_step(agent, obs, z)
            planned_now = exploit and a == THINK \
                and env.agent_state != env.goal
            prev_z = z
            obs, r, done, _, info = env.step(a)
            prev_planned = planned_now and info.get("planned", False)
            if r > 0.5:
                exploit = True
    return {k: np.array(v, dtype=np.float64) for k, v in res.items()}


def pca_cosines(res):
    '''PCA-space cosines between alpha_RNN variants and alpha_PG.'''
    def norm_rows(A):
        A = A - A.mean(axis=0, keepdims=True)
        return A / np.linalg.norm(A, axis=1, keepdims=True)

    def cos_for(jacs, grads, sims):
        betas = np.stack([grads[i, :, int(s)] for i, s in enumerate(sims)])
        jn, bn = norm_rows(jacs), norm_rows(betas)
        pca = PCA(n_components=3).fit(bn)
        Za = pca.transform(jn)
        Zb = pca.transform(bn)
        Za /= np.linalg.norm(Za, axis=1, keepdims=True)
        Zb /= np.linalg.norm(Zb, axis=1, keepdims=True)
        return float(np.mean(np.sum(Za * Zb, axis=1)))

    has2 = res["sim_a2s"] >= 0
    cos1 = [cos_for(res["jacs"], res["gs"], res["sim_as"]),
            cos_for(res["jacs_shift"], res["gs"], res["sim_as"])]
    cos2 = [cos_for(res["jacs"][has2], res["gs2"], res["sim_a2s"][has2]),
            cos_for(res["jacs_shift2"], res["gs2"], res["sim_a2s"][has2])]
    return cos1, cos2


def _fig5():
    r_tau = estimate_r_tau()
    res = pg_alignment(r_tau)
    return {"r_tau": r_tau, "res": res}


F5 = cached(os.path.join(EVAL_DIR, "fig5.pkl"), _fig5)
res5 = F5["res"]
cos1, cos2 = pca_cosines(res5)
print(f"{len(res5['jacs'])} rollouts analyzed, "
      f"{int((res5['sim_a2s'] >= 0).sum())} with >= 2 imagined actions")
print(f"cos(alpha_RNN, alpha_PG) 1st action: true {cos1[0]:.3f} | "
      f"permuted {cos1[1]:.3f}")
print(f"cos(alpha_RNN, alpha_PG) 2nd action: true {cos2[0]:.3f} | "
      f"permuted {cos2[1]:.3f}")

fig, ax = plt.subplots(figsize=(5.2, 3.6))
x = np.arange(2)
ax.bar(x - 0.18, [cos1[0], cos2[0]], width=0.36, color="steelblue",
       label="true feedback")
ax.bar(x + 0.18, [cos1[1], cos2[1]], width=0.36, color="lightgray",
       label="permuted-action control")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x, ["1st imagined action", "2nd imagined action"])
ax.set_ylabel(r"cos($\alpha_{RNN}$, $\alpha_{PG}$) in PC space")
ax.set_title("rollout feedback aligns with policy gradient")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig5_pg_alignment.png"))
plt.show()

The alignment is real and specific. For the first imagined
action the cosine between what the feedback does to the hidden state and
what a policy-gradient step for that action would do is ~0.40 — and it
collapses below zero when the same feedback vector is delivered with the
first action permuted, so the effect is carried by the *content*, not the
magnitude, of the feedback. The alignment persists essentially undiminished
(~0.40) for the second imagined action — the feedback pushes the hidden
state along a multi-step plan, not just the next move. In the paper's
framing: the recurrent dynamics have learned to use replay events as
approximate policy-improvement steps, which is precisely the function
proposed for hippocampal replay in model-based RL.

## 11. The world model improves over training (supplementary)

Finally, a sanity check on the auxiliary head itself, across training. For the checkpoints saved
across training we measure two accuracies on
fresh greedy episodes: the next-**state** prediction (argmax of the state
slice vs. the state the agent actually lands in, skipping teleport steps)
and the **reward-location** prediction (argmax of the reward slice vs. the
true goal, exploitation phase only — before the first reward the goal is
unknowable, so accuracy there is not meaningful).

In [ ]:
# --- world-model accuracy per checkpoint ------------------------------------
import re as _re


def world_model_accuracy(agent_ck, eps_ck):
    '''(next-state accuracy, reward-location accuracy) on eval episodes.'''
    st_ok = st_n = rw_ok = rw_n = 0
    for ep in eps_ck:
        z = torch.as_tensor(ep["states"])
        a = torch.as_tensor(ep["actions"], dtype=torch.long)
        with torch.no_grad():
            aux = agent_ck.aux_logits(z, a).numpy()
        spred = aux[:, STATE_SLICE[0]:STATE_SLICE[1]].argmax(axis=1)
        rpred = aux[:, REWARD_SLICE[0]:REWARD_SLICE[1]].argmax(axis=1)
        rews = ep["rewards"]
        targets = np.array([si["aux_targets"] for si in ep["step_infos"]])
        goal = np.array([si["goal_loc"] for si in ep["step_infos"]])
        exploit = False
        for t in range(len(rews)):
            if not exploit and rews[t] > 0.5:
                exploit = True
                continue            # next step is a teleport: skip it
            st_ok += int(spred[t] == targets[t, 0])
            st_n += 1
            if exploit:
                rw_ok += int(rpred[t] == goal[t])
                rw_n += 1
    return st_ok / max(st_n, 1), rw_ok / max(rw_n, 1)


def _figS():
    '''World-model accuracy across training (saved checkpoints).'''
    names = [d for d in os.listdir(RUN_DIR) if d.startswith("checkpoint")]
    names.sort(key=lambda d: int(_re.search(r"\d+", d).group()))
    rows = []
    for name in names[::2]:              # every second checkpoint is plenty
        agent_ck = AutoModel.from_pretrained(
            os.path.join(RUN_DIR, name)).eval()
        planner_ck = WorldModelPlanner(
            plan_depth=PLAN_DEPTH, n_move_actions=NMOVE,
            state_slice=STATE_SLICE, reward_slice=REWARD_SLICE,
            seed=0).bind(agent_ck)
        eps_ck = collect_eval_episodes(agent_ck, planner_ck, n_episodes=50)
        sa, ra = world_model_accuracy(agent_ck, eps_ck)
        upd = int(_re.search(r"\d+", name).group())
        rows.append((upd * 6800 / 1e6, sa, ra))   # global update -> M steps
        print(f"  {name}: state {sa:.3f}, reward {ra:.3f}")
    return rows


rows = cached(os.path.join(EVAL_DIR, "figS_wm.pkl"), _figS)
upd = np.array([r[0] for r in rows])
sa = np.array([r[1] for r in rows])
ra = np.array([r[2] for r in rows])

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(upd, sa, "o-", color="steelblue", ms=4, lw=1.6,
        label="next-state accuracy")
ax.plot(upd, ra, "s-", color="darkorange", ms=4, lw=1.6,
        label="reward-location accuracy (exploitation)")
ax.set_xlabel("environment steps (millions)")
ax.set_ylabel("world-model accuracy")
ax.set_ylim(0.4, 1.02)
ax.legend(fontsize=8, frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "figS_worldmodel_training.png"))
plt.show()

The next-state prediction saturates within the first few
million steps — the single-step physics of the maze is easy to learn — and
the reward-location prediction (measured in exploitation, where the goal
has been seen once) follows shortly after. Both then sit at ceiling for
the rest of training, while the behavioral return keeps climbing for the
whole run. This is worth stating plainly because it clarifies where the
difficulty of the task lives: not in *representing* the maze, but in
learning to plan with it. It also underwrites the Fig 4 wall-conformity
result: the imagination respects walls because the world model that
generates it is essentially exact.
